# Repurchase Cycle Analysis - When should an email Win Back flow start?

## About this project
**The question:** is a fixed Win Back threshold of 210 days since the last order a good fit for how customers actually repurchase?

I analyze the order history of this D2C apparel e-commerce store (WooCommerce) to answer it. The store uses two kinds of marketing emails:
- a **flow**: an automated sequence of emails in Klaviyo that starts when a customer has not bought for 210 days;
- **broadcasts**: one-off emails sent to a whole audience, each one with its own occasion.

## What I noticed
**1. Inactive customers converted more than I expected.** While reviewing the email results, I compared how the same broadcast performed on three audiences. In September-October 2025, the store sent the same content in the same week to active customers, to Win Back + Churn Risk customers (no purchase for 210+ days) and to subscribers who had never bought. The Win Back + Churn Risk audience converted almost 6 times more than the subscribers without purchases, and about one third as much as the active customers. For an audience that is supposed to be inactive, this is more than I expected. I repeated the test in 2026 and the results confirmed the same pattern.

**2. Some customers kept buying after the flow had stopped writing to them.** They came back after a broadcast, months after the end of the Win Back flow.

## Three possible explanations
1. **Timing and occasion:** a broadcast always comes with a concrete reason to buy (a seasonal change, a wardrobe change, new arrivals). On top of that, it either reminds the customer of the 10% discount they already had in the flow, or it offers a new discount of the same percentage. The flow, instead, starts at a moment that depends only on when the customer reached 210 days without buying.
2. **Reminder effect:** the flow already moved the customer towards the purchase, and the broadcast closes it.
3. **A longer repurchase cycle:** many customers naturally repurchase after more than 210 days, so the flow reaches them before they need it.

## What this notebook does
It focuses on the third explanation and, in part, on the second.

**Why I did not evaluate the first explanation:** all the broadcasts sent to this Win Back + Churn Risk audience have a concrete reason to buy. So I do not have broadcasts without a reason to buy, and therefore no control group to compare them with.

My steps:
1. I measure how many days pass between two consecutive orders of the same customer.
2. I look at the distribution of those intervals and I compare it with the 210-day threshold.
3. I check when Win Back customers come back after one of the 10 email broadcasts where I noticed this pattern.
4. I finish with recommendations.

## About the data
The data comes from a real store. Before publishing I removed all personal data in a private notebook: customers are replaced by random IDs, and I kept only five columns (order id, order date, order total, anonymous customer ID and coupon items). I also kept only completed orders. The raw export is not published.

## 1. Load the anonymized data
I load the checkpoint I created in the private notebook.

In [0]:
import pandas as pd

# I load the anonymized checkpoint created in the private notebook
df = pd.read_csv("../data/processed/orders_anonymized.csv")
df.shape

## 2. Prepare the dates
`order_date` is still text, so I convert it to a real date. I first check if there are missing dates.

In [0]:
df["order_date"].isna().sum()

In [0]:
# I convert order_date to a real datetime, using coerce so invalid dates become Null instead of crashing
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

In [0]:
# I check how many dates failed to convert
df["order_date"].isna().sum()

In [0]:
df["order_date"].sample(10)

## 3. Orders and customers tables
I create an orders table (one row per order) and a customers table (one row per customer). I use the customers table to check that the number of customers is correct.

In [0]:
# My orders table is just the current df: one row per order, already linked by customer_id
orders = df.copy()

In [0]:
orders.shape

In [0]:
# I make sure order_total is numeric, since I'll sum it in the customers table
df["order_total"] = pd.to_numeric(df["order_total"], errors="coerce")

In [0]:
# I build the customers table by aggregating orders per customer_id
customers = df.groupby("customer_id").agg(
    n_orders=("order_id", "count"),
    first_order_date=("order_date", "min"),
    last_order_date=("order_date", "max"),
    total_spent=("order_total", "sum"),
).reset_index()

In [0]:
customers.shape

## 4. Repurchase cycle
### Days since the previous order
For each order I calculate how many days passed since the previous order of the same customer. The first order of each customer has no previous order, so it gets a missing value.

In [0]:
# I sort orders by customer and date, so consecutive orders of the same customer are next to each other
orders_sorted = orders.sort_values(["customer_id", "order_date"])

# I calculate the number of days since the previous order of the SAME customer
# groupby + diff() makes sure I don't compare across different customers
orders_sorted["days_since_prev_order"] = orders_sorted.groupby("customer_id")["order_date"].diff().dt.days

# I look at the result
orders_sorted[["customer_id", "order_date", "days_since_prev_order"]].head(20)
# Output: for each row, how many days passed since the previous order of the same customer
# the first row of each customer will be NaN, because it has no previous order to compare it with

### Coupon used in each order
I extract the coupon code from the coupon text of each order, so I can tell later which orders used a coupon.

In [0]:
# I import re to extract the coupon code with a regular expression
import re

# I define a function that extracts the coupon code from the coupon_items text
def extract_coupon_code(coupon_items):
    # If there's no coupon, I return None instead of crashing
    if pd.isna(coupon_items) or coupon_items == "":
        return None
    # I search for the pattern "code:something" and stop at the next "|"
    match = re.search(r"code:([^|]+)", str(coupon_items))
    # I return the matched code, or None if the pattern isn't found
    return match.group(1) if match else None

# I create a new column with just the coupon code
orders_sorted["coupon_code"] = orders_sorted["coupon_items"].apply(extract_coupon_code)

# I check how many orders actually used a coupon
print(orders_sorted["coupon_code"].notna().sum())

### Repurchase intervals
I keep only the rows where an interval exists. These are the orders of customers who bought at least twice.

In [0]:
# I keep only the rows where an interval actually exists (drop the NaN from first purchases)
intervals = orders_sorted["days_since_prev_order"].dropna()

# I check how many repurchase intervals I have in total
print(len(intervals))
# Output: number of intervals I can calculate (only customers with 2+ orders)

### How long do customers wait before buying again?
I calculate the median and the 75th and 80th percentiles. Note that I count repurchase **events**, not unique customers: a loyal customer with many orders contributes several intervals.

In [0]:
# I calculate the key percentiles once, so I can reuse them in the sentences
p50 = intervals.median()
p75 = intervals.quantile(0.75)
p80 = intervals.quantile(0.80)

# I print each slice of repurchase intervals with its specific day range
# NOTE: this counts repurchase EVENTS, not unique customers, so a loyal customer with many orders contributes multiple intervals
print(f"50% of repurchase intervals are within {p50:.0f} days.")
print(f"25% of repurchase intervals fall between {p50:.0f} and {p75:.0f} days.")
print(f"5% of repurchase intervals fall between {p75:.0f} and {p80:.0f} days.")
print(f"The remaining 20% of repurchase intervals take more than {p80:.0f} days.")

### Distribution of repurchase intervals
I plot the intervals and I add the percentiles and the 210-day Win Back threshold. I cap the chart at 400 days so extreme outliers do not squash it.

In [0]:
# I import matplotlib to draw the chart
import matplotlib.pyplot as plt

# I create the figure and set its size
plt.figure(figsize=(10, 6))

# I plot the histogram of repurchase intervals, capping at 400 days so extreme outliers don't squash the chart
plt.hist(intervals[intervals <= 400], bins=50, color="#4C72B0", edgecolor="white")

# I add a vertical line for the median
plt.axvline(p50, color="green", linestyle="--", label=f"Median: {p50:.0f} days")

# I add a vertical line for the 75th percentile
plt.axvline(p75, color="orange", linestyle="--", label=f"75th percentile: {p75:.0f} days")

# I add a vertical line for the 80th percentile
plt.axvline(p80, color="red", linestyle="--", label=f"80th percentile: {p80:.0f} days")

# I add a vertical line for the Win Back threshold (210 days), to compare it against the distribution
plt.axvline(210, color="black", linestyle="-", label="Win Back threshold: 210 days")

# I add titles and labels
plt.title("Distribution of repurchase intervals")
plt.xlabel("Days since previous order")
plt.ylabel("Number of orders")
plt.legend()

# I display the chart
plt.show()

### What these numbers say
- **The distribution is very wide.** There is no single repurchase cycle for this store. Half of the repurchases happen within 4 months (121 days), but a fifth (20%) happen after 343 days, almost a year. With this spread, one fixed threshold, like the 210 days of the Win Back flow, cannot work well for everyone.
- **210 days threshold falls in a "dead zone"**, between 121 and 283 days. It is not the moment when most customers repurchase (that is earlier, around 121 days), and it is not the moment when the slow tail wakes up (that is later, around 280-340 days). It does not catch well either the "normal" customers or the "slow" ones.
- **The 20% beyond 343 days is a huge group that I should not treat as "at risk".** It is mostly made of customers with a naturally long cycle, for example one purchase a year, linked to a seasonal change. The Win Back segment still triggers at 210 days for everyone, so it labels them "at risk of churn" and sends them the 10% discount months before they really need it. I waste the discount at a moment when the need to buy is not there yet.
- **The flow probably "burns" the discount too early** for a large group of people. Someone with a natural cycle of 280-350 days gets the 10% discount at 210 days, when they are not ready to buy yet. If a broadcast arrives months later (like the 10 email broadcasts selected in this analysis), closer to their real repurchase moment, that email converts better. The reason is not better content, but timing closer to the real need.

### Recommendation
I think it is worth testing a split of the Win Back segment by the customer's historical cycle (for example customers with a typical cycle under 150 days vs customers with a typical cycle over 250 days), instead of one 210-day threshold for everyone. A customer who always repurchased quickly can really be "at risk" at 210 days. A customer who always repurchased slowly probably is not.

## 5. Reality check: when do Win Back customers come back after a broadcast?
I take all the customers who:
- had not bought for at least 247 days (8+ months). I use 247 days because the Win Back flow starts at 210 days of inactivity and lasts 37 days (210 + 37 = 247). After 247 days, the customer has received all the emails of the flow and the flow has stopped writing to them. So a purchase after that point is not a direct reply to an email of the flow;
- received one of the 10 Win Back + Churn Risk broadcast emails;
- and bought within 5 days after receiving on of these 10 broadcast emails using a coupon.

For this group I measure one thing: how much time had passed since their previous order?

### Why it matters
If these customers wait on average 450-500 days before buying again, it means the automated flow, which starts at only 210 days, reaches them too early. The flow gives up on them after 247 days (210 days + 37 days of flow duration), but they come back much later. It is the broadcast, sent months later, that reaches them at the right moment.

In [0]:
# I create a small table with the broadcast send dates I found via the Klaviyo API
broadcast_dates = pd.to_datetime([
    "2025-02-21", "2025-09-08", "2025-09-17", "2025-09-24", "2025-10-01", "2025-10-08",
    "2025-10-15", "2025-10-22", "2025-10-29", "2026-09-22"
])

In [0]:
# I set the attribution window in days
window_days = 5

In [0]:
# I set the minimum inactivity threshold: 210 days Win Back entry + 37 days flow duration
min_days_threshold = 247

### Step 1: orders inside the attribution window
Among all orders, I keep only the ones that fall after an email sending, inside the attribution window (5 days after the sending).

In [0]:
# I create an empty list where I will collect matching orders
broadcast_orders_list = []

# I loop through each broadcast date, one at a time
for send_date in broadcast_dates:
    # I calculate the last day of the attribution window for this broadcast (sending date + 5 days attribution window)
    window_end = send_date + pd.Timedelta(days=window_days)

    # I loop through every single order in my orders_sorted table
    for index, row in orders_sorted.iterrows():
        # I check if this order's date falls inside the attribution window
        order_is_in_window = row["order_date"] >= send_date and row["order_date"] <= window_end

        # If it's inside the window, I keep this order
        if order_is_in_window:
            # I copy the row so I can add a new piece of information to it
            row_copy = row.copy()
            # I record which broadcast this order belongs to
            row_copy["broadcast_date"] = send_date
            # I add this row to my list of matches
            broadcast_orders_list.append(row_copy)

# I turn my list of matching rows into a proper table
broadcast_orders = pd.DataFrame(broadcast_orders_list)

# I check how many orders I found in total
print(len(broadcast_orders))

### Step 2: strict filters
Among the orders inside the attribution window, I keep only the ones that:
- come from Win Back customers who already finished the flow without purchasing (210 days to become a Win Back customer + 37 days of flow),
- come from customers who used a coupon).

In [0]:
# I create an empty list where I will collect only the orders that pass ALL three checks (inactive>=247 days, has used a coupon (loyalty points excluded))
final_rows = []

# I loop through every order I found in the previous step
for index, row in broadcast_orders.iterrows():

    # CHECK 1: was the customer inactive for at least 247 days?
    check_1_inactive_enough = row["days_since_prev_order"] >= min_days_threshold

    # CHECK 2: did this order actually use a coupon? (if coupon_code is missing, this is False)
    check_2_has_coupon = pd.notna(row["coupon_code"])

    # CHECK 3: if there IS a coupon, make sure it's not a loyalty points code
    # I only run this check if check_2 passed, otherwise there's no coupon to check
    if check_2_has_coupon:
        check_3_not_loyalty = not row["coupon_code"].startswith("points_redemption")
    else:
        check_3_not_loyalty = False

    # I keep this row only if ALL three checks are True
    if check_1_inactive_enough and check_2_has_coupon and check_3_not_loyalty:
        final_rows.append(row)

# I turn my list of final rows into a proper table
final_subset = pd.DataFrame(final_rows)

# I check how many orders are left after all filters
print(len(final_subset))

### Results
I calculate the median and the 75th and 80th percentiles of the inactivity of these customers.

In [0]:
# I calculate the median: the middle value, meaning half of these customers waited less, half waited more
final_median = final_subset["days_since_prev_order"].median()

# I calculate the 75th percentile: the point below which 3 out of 4 customers fall
final_p75 = final_subset["days_since_prev_order"].quantile(0.75)

# I calculate the 80th percentile: the point below which 4 out of 5 customers fall
final_p80 = final_subset["days_since_prev_order"].quantile(0.80)

# I print the results:
print(f"Out of {len(final_subset)} orders analysed:")
print(f"- Half came from customers inactive for at least {final_median:.0f} days.")
print(f"- 1 order out of 4 came from customers inactive for at least {final_p75:.0f} days.")
print(f"- 1 order out of 5 came from customers inactive for at least {final_p80:.0f} days.")

In [0]:
# I create the figure
plt.figure(figsize=(10, 6))

# I plot the final subset's distribution, capped at 800 days for readability
plt.hist(final_subset["days_since_prev_order"].clip(upper=800), bins=30,
         color="#C44E52", edgecolor="white", alpha=0.8)

# I add a vertical line for the Win Back threshold (210 days)
plt.axvline(210, color="black", linestyle="-", label="Win Back threshold: 210 days")

# I add a vertical line for the median
plt.axvline(final_median, color="darkred", linestyle="--", label=f"Median: {final_median:.0f} days")

# I add titles and labels
plt.title("After how many days of inactivity do Win Back customers come back after a broadcast? - strict attribution (>=247 days, coupon used, loyalty excluded)")
plt.xlabel("Days since previous order")
plt.ylabel("Number of orders")
plt.legend()

# I display the chart
plt.show()

### Note on the chart
The empty space between 210 and 270 days does not mean there were no purchases in that range. It comes from the filter I applied (`days_since_prev_order >= 247`), which deliberately excludes all orders placed before 247 days of inactivity. The chart only shows customers who passed that threshold. It says nothing about what happens between 210 and 247 days.

### What the chart shows, in detail
- **270-360 days (9-12 months):** the most populated group, with peaks of 4-5 orders per bar. These are customers who waited between 9 months and a year.
- **370-470 days (12-15.5 months):** lower bars (1-3 orders each), fairly evenly spread. No particular concentration.
- **500-520 days (about 16.5-17 months):** a small rise (4 orders), exactly where the dashed median line falls (505 days).
- **520-600 days (17-20 months):** an almost empty zone, very few orders. A "hole" in the distribution.
- **600-700 days (20-23 months):** it rises a little again (1-3 orders per bar), with a peak of 3 around 680 days.
- **700-780 days (23-26 months):** low again, 1 order per bar.
- **About 790-800 days (more than 2 years):** the highest bar of the whole chart, 7 orders, at the far right. This bar is artificially inflated. I capped the chart at 800 days, so every order placed after 800 days, whether 810 or 1300, is forced into this last bar. It does not mean there is a real peak of customers returning at exactly 800 days. It means that all the orders of customers who waited 800 days or more pile up there.

### Key points from the chart
Among the 51 orders attributed to the broadcasts, customer inactivity goes from about 265 days (9 months) to more than 800 (over 2 years), spread fairly evenly. Half of the customers had been inactive for less than 505 days, the other half for more.

Even customers who had been away for 1.5-2 years or more come back to buy when they receive an email with a discount. About 1 order in 7 (the bar on the right) comes from customers inactive for more than 800 days. So stopping to contact them after the flow, at 247 days, means giving up part of these purchases. This is the data that supports a multi-step flow.

The longest bars show after how many days Win Back customers who receive a broadcast with a coupon tend to buy. However, the data is very small (51 orders) to set thresholds. What I can say is that there seem to be groups of customers with a repurchase cycle much longer than 210 days, and it is worth testing this hypothesis.

## 6. Recommendations

### Premise
The analysis of 51 orders shows that Win Back customers tend to repurchase well beyond 210 days. But the sample is small and I do not have a comparison group (Win Back customers who never received any email). I take this as a hypothesis to verify, not as a final result. The actions below are meant to verify it.

### 1. Turn the Win Back flow into a multi-step flow
- I would keep the first email at 210 days and add later steps at regular intervals, for example at 10, 14, 18 and 24 months after the last purchase.
- I would not tie the steps to the peaks I see in the data: with this sample, they are mostly random.
- Where possible, I would link the content to a concrete occasion, the format that has worked best in the broadcasts so far.
- The discount can be a percentage or an amount in euros (both seem to work). I would keep a percentage discount in the form of a coupon, for consistency.

### 2. Use a different discount code for each step and for the broadcasts
- For each step of the flow and for each broadcast, I would create in Klaviyo a coupon with a different prefix (for example `WB07-` for the 7-month step, `WB10-` for the 10-month step, `BC-` for the broadcasts).
- Today the flow and the broadcasts share the same code, so it is impossible to know which of the two generated the purchase.

### 3. Keep a control group
- I would exclude a random 10% of Win Back customers from the flow, to use as a comparison group. It is the only way to know how many of them would repurchase anyway, without receiving emails.

### 4. Protect deliverability
- After the first 2 or 3 steps, I would send only to customers who opened or clicked at least one email in the last 6-12 months, to avoid deliverability problems.

### 5. Measure per recipient, with rules decided in advance
- For each step: percentage of customers who repurchase and revenue per recipient net of the discount, compared with the control group.
- A fixed attribution window, the same for all steps (for example 5 days after sending).
- First review of the results after 6 months, then I would remove the steps that do not convert more than the control group.

## Presentation
To see all the results and the recommendations in one place, I prepared a presentation, available in this repository: [`repurchase_cycle_presentation.pdf`](repurchase_cycle_presentation.pdf).